In [2]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

provider = TracerProvider()
provider.add_span_processor(
    SimpleSpanProcessor(ConsoleSpanExporter())
)
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

NameError: name 'SQLiteSpanExporter' is not defined

In [2]:
from starter import rag

## Q1

In [3]:
from rag_helper import RAGBase

class RAGTraced(RAGBase):
    def rag(self,query):
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)      
            return super().rag(query)
    
    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)
            return super().search(query)
    
    def llm(self, query):
        with tracer.start_as_current_span("llm") as span:
            span.set_attribute("query", query)
            return super().llm(query)
    
    

In [6]:
rag_traced = RAGTraced(
    index=rag.index,
    llm_client=rag.llm_client,
    instructions=rag.instructions,
    prompt_template=rag.prompt_template,
    model=rag.model,
)

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag_traced.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x47c3d34c802ee568565e7b608fcf83d8",
        "span_id": "0xa262bab177536ccc",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xd839e3cdc0a9ec93",
    "start_time": "2026-07-20T18:02:31.028653Z",
    "end_time": "2026-07-20T18:02:31.038183Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "306f9fdf-534e-4f49-ae6c-367e20afd3e8",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x47c3d34c802ee568565e7b608fcf83d8",
        

## Q2

In [9]:
class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)

            return super().search(
                query,
                num_results=num_results,
            )

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)

            usage = response.usage

            span.set_attribute(
                "input_tokens",
                usage.input_tokens,
            )
            span.set_attribute(
                "output_tokens",
                usage.output_tokens,
            )

            input_cost = (
                usage.input_tokens * 0.15 / 1_000_000
            )
            output_cost = (
                usage.output_tokens * 0.60 / 1_000_000
            )
            cost = input_cost + output_cost

            span.set_attribute("cost", cost)

            return response

In [10]:
rag_traced = RAGTraced(
    index=rag.index,
    llm_client=rag.llm_client,
    instructions=rag.instructions,
    prompt_template=rag.prompt_template,
    model=rag.model,
)

In [11]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag_traced.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x4a0dd10748e6f4a8a55e17d5f4d58db7",
        "span_id": "0x1387bdfd7df59ca9",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x3b6b40437ec71c51",
    "start_time": "2026-07-20T18:09:52.075757Z",
    "end_time": "2026-07-20T18:09:52.083297Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "query": "How does the agentic loop keep calling the model until it stops?"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "306f9fdf-534e-4f49-ae6c-367e20afd3e8",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x4a0dd10748e6f4a8a55e17d5f4d58db7",
        

## Q4

In [1]:
import sqlite3
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True

In [2]:
provider = TracerProvider()

provider.add_span_processor(
    SimpleSpanProcessor(SQLiteSpanExporter("traces.db"))
)

trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [4]:
from starter import rag
from rag_helper import RAGBase
class RAGTraced(RAGBase):
    def rag(self, query):
        with tracer.start_as_current_span("rag") as span:
            span.set_attribute("query", query)
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            span.set_attribute("query", query)

            return super().search(
                query,
                num_results=num_results,
            )

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage

            span.set_attribute(
                "input_tokens",
                usage.input_tokens,
            )
            span.set_attribute(
                "output_tokens",
                usage.output_tokens,
            )

            input_cost = (
                usage.input_tokens * 0.15 / 1_000_000
            )
            output_cost = (
                usage.output_tokens * 0.60 / 1_000_000
            )
            cost = input_cost + output_cost

            span.set_attribute("cost", cost)

            return response

In [9]:
rag_traced = RAGTraced(
    index=rag.index,
    llm_client=rag.llm_client,
    instructions=rag.instructions,
    prompt_template=rag.prompt_template,
    model=rag.model,
)

In [12]:
query = "How does the agentic loop keep calling the model until it stops?"

answer = rag_traced.rag(query)
print(answer)

It keeps calling the model inside a `while True` loop.

After each model response, the code checks whether the response contains any `function_call` items:

- if yes, it runs the tool, appends the tool result to `messages`, and loops again
- if no, it breaks out of the loop

So the stop condition is פשוט: **no function calls in the latest response**.


## Q5

In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("traces.db")

duration_df = pd.read_sql_query(
    """
    SELECT
        name,
        COUNT(*) AS span_count,
        SUM(end_time - start_time) / 1000000000.0
            AS total_duration_seconds
    FROM spans
    WHERE name != 'rag'
    GROUP BY name
    ORDER BY total_duration_seconds DESC
    """,
    conn,
)

duration_df

,name,span_count,total_duration_seconds
0,llm,3,7.482332
1,search,3,0.014792


## Q6

In [13]:
token_df = pd.read_sql_query(
    """
    SELECT
        rowid,
        input_tokens
    FROM spans
    WHERE name = 'llm'
    GROUP BY rowid
    """,
    conn,
)

token_df

,rowid,input_tokens
0,2,7111
1,5,7111
2,8,7111
3,11,7111
